In [1]:
import subprocess, base64, os, glob, shutil
from kaggle_secrets import UserSecretsClient

REPO_URL = "https://github.com/Rosierobber/Temporal-Graph-ML-Project.git"
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/Temporal-Graph-ML-Project"

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()

def git(*args, cwd=REPO_DIR):
    r = subprocess.run(["git", "-c", f"http.extraheader={hdr}", *args],
                       cwd=cwd, capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, "***"))
    return r.returncode

def save_and_push(msg, paths=("notes", "results", "logs", "README.md")):
    git("add", *paths)
    git("status", "--short")
    git("commit", "-m", msg)
    git("pull", "--rebase", "origin", "main")
    git("push", "origin", "main")

if not os.path.exists(REPO_DIR):
    git("clone", REPO_URL, cwd=WORK)
git("config", "user.name", "Rabia Habib")
git("config", "user.email", "rabihabib0411@gmail.com")
git("pull", "--rebase", "origin", "main")
git("log", "--oneline", "-3")

Cloning into 'Temporal-Graph-ML-Project'...



Already up to date.
From https://github.com/Rosierobber/Temporal-Graph-ML-Project
 * branch            main       -> FETCH_HEAD

99c3109 Fig 3: sensitivity to window ratio
3d36290 R-EdgeBank on tgbl-review
e834285 EdgeBank on tgbl-review



0

In [2]:
!pip install -q py-tgb torch_geometric
import tgb
print("ok")

ok


In [3]:
!pip install -q py-tgb torch_geometric
import torch, subprocess
subprocess.run("pip install -q torch_scatter -f https://data.pyg.org/whl/torch-2.11.0+cu130.html", shell=True)
from torch_scatter import scatter_max
x = torch.randn(10, device="cuda"); idx = torch.zeros(10, dtype=torch.long, device="cuda")
print("torch_scatter OK on GPU:", scatter_max(x, idx)[0].shape)

def save_and_push(msg, paths=("notes", "results", "logs", "README.md")):
    git("add", *paths); git("status", "--short"); git("commit", "-m", msg)
    git("pull", "--rebase", "--autostash", "origin", "main"); git("push", "origin", "main")

TGB = f"{REPO_DIR}/third_party/TGB"
!cd {TGB} && PYTHONPATH=. python examples/linkproppred/tgbl-review/dyrep.py --help | head -25

torch_scatter OK on GPU: torch.Size([1])
/usr/local/lib/python3.13/dist-packages/clint/textui/colored.py:118: SyntaxWarning: invalid escape sequence '\$'
  strip = re.compile("([^-_a-zA-Z0-9!@#%&=,/'\";:~`\$\^\*\(\)\+\[\]\.\{\}\|\?\<\>\\]+|[^\s]+)")
/usr/local/lib/python3.13/dist-packages/clint/packages/colorama/ansitowin32.py:43: SyntaxWarning: invalid escape sequence '\['
  ANSI_RE = re.compile('\033\[((?:\d|;)*)([a-zA-Z])')
/usr/local/lib/python3.13/dist-packages/clint/textui/prompt.py:68: SyntaxWarning: "is not" with 'str' literal. Did you mean "!="?
  if prompt[-1] is not ' ':
usage: *** TGB *** [-h] [-d DATA] [--lr LR] [--bs BS] [--k_value K_VALUE]
                   [--num_epoch NUM_EPOCH] [--seed SEED] [--mem_dim MEM_DIM]
                   [--time_dim TIME_DIM] [--emb_dim EMB_DIM]
                   [--tolerance TOLERANCE] [--patience PATIENCE]
                   [--num_run NUM_RUN]

options:
  -h, --help            show this help message and exit
  -d, --data DATA       Datas

In [4]:
!pip install -q py-tgb torch_geometric
import sys, torch, platform, importlib.metadata as md
def v(p):
    try: return md.version(p)
    except Exception: return "n/a"
info = [f"python {sys.version.split()[0]}", f"torch {torch.__version__}",
        f"torch_geometric {v('torch_geometric')}", f"py-tgb {v('py-tgb')}", f"platform {platform.platform()}"]
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip()
os.makedirs(f"{REPO_DIR}/logs", exist_ok=True)
open(f"{REPO_DIR}/logs/environment_accountB.txt", "w").write("\n".join(info) + "\nGPUs:\n" + gpu + "\n")
print(open(f"{REPO_DIR}/logs/environment_accountB.txt").read())
save_and_push("Record environment of second Kaggle session", ["logs/environment_accountB.txt"])

python 3.13.16
torch 2.11.0+cu130
torch_geometric 2.8.1
py-tgb 2.3.0
platform Linux-6.18.55+-x86_64-with-glibc2.39
GPUs:
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


A  logs/environment_accountB.txt

[main 2c126b2] Record environment of second Kaggle session
 1 file changed, 8 insertions(+)
 create mode 100644 logs/environment_accountB.txt

Current branch main is up to date.
From https://github.com/Rosierobber/Temporal-Graph-ML-Project
 * branch            main       -> FETCH_HEAD

To https://github.com/Rosierobber/Temporal-Graph-ML-Project.git
   0821005..2c126b2  main -> main



In [5]:
BASE_URL    = "https://github.com/fpour/TGB_Baselines.git"
BASE_COMMIT = "d2022fae4a3311185f611bab733e267938251f42"
BASE        = f"{REPO_DIR}/third_party/TGB_Baselines"

if not os.path.exists(f"{BASE}/train_tgb_lpp_original.py"):
    os.makedirs(os.path.dirname(BASE), exist_ok=True)
    subprocess.run(["git", "clone", "-q", BASE_URL, BASE], check=True)
    subprocess.run(["git", "checkout", "-q", BASE_COMMIT], cwd=BASE, check=True)

actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=BASE, text=True).strip()
print("TGB_Baselines commit:", actual)
assert actual == BASE_COMMIT, "Wrong TGB_Baselines commit"

TGB_Baselines commit: d2022fae4a3311185f611bab733e267938251f42


In [6]:
import threading, time, re

EPOCHS = 5                 # must match seeds 1 and 2
SEEDS  = [3, 4]
RES = f"{REPO_DIR}/results/graphmixer"
os.makedirs(RES, exist_ok=True); os.makedirs(f"{REPO_DIR}/logs", exist_ok=True)

def result_path(s): return f"{RES}/seed{s}_GraphMixer_tgbl-wiki.json"
def log_path(s):    return f"{REPO_DIR}/logs/graphmixer_wiki_seed{s}.txt"

def launch(s, gpu):
    cmd = (f"cd {BASE} && python train_tgb_lpp_original.py --dataset_name tgbl-wiki "
           f"--model_name GraphMixer --num_epochs {EPOCHS} --num_runs 1 --seed {s} --gpu {gpu} "
           f"> {log_path(s)} 2>&1")
    return subprocess.Popen(cmd, shell=True)

def progress(seeds):
    print(f"--- progress at {time.strftime('%H:%M:%S')} ---")
    for s in seeds:
        p = log_path(s)
        if not os.path.exists(p): print(f"seed {s}: no log yet"); continue
        t = open(p, errors="ignore").read().replace("\r", "\n")
        b = re.findall(r"train for the (\d+)-th batch, train loss: ([0-9.]+)", t)
        print(f"seed {s}: " + (f"batch {b[-1][0]}, loss {b[-1][1]}" if b else "no batch lines yet"))

def finish(s):
    if "Test: mrr" not in open(log_path(s), errors="ignore").read():
        print(f"seed {s}: no final test MRR, not saved. Check {log_path(s)}"); return
    src = glob.glob(f"{BASE}/saved_results/GraphMixer/tgbl-wiki*/GraphMixer_tgbl-wiki_seed_{s}_run_0*.json")
    if src: shutil.copy(src[0], result_path(s))
    else:   print(f"seed {s}: no result JSON found; log is still saved")
    save_and_push(f"GraphMixer tgbl-wiki seed {s} ({EPOCHS} epochs, second session)",
                  [f"logs/graphmixer_wiki_seed{s}.txt", "results/graphmixer"])

def run_all():
    pending = [s for s in SEEDS if not os.path.exists(result_path(s))]
    print("seeds to run:", pending)
    running, free_gpus, last_report = {}, [0, 1], 0
    while pending or running:
        while pending and free_gpus:
            s, g = pending.pop(0), free_gpus.pop(0)
            running[s] = (launch(s, g), g)
            print(f"started seed {s} on GPU {g} at {time.strftime('%H:%M:%S')}")
            time.sleep(180)
        for s, (p, g) in list(running.items()):
            if p.poll() is not None:
                print(f"seed {s} ended (exit {p.returncode}) at {time.strftime('%H:%M:%S')}")
                finish(s); free_gpus.append(g); del running[s]
        if running and time.time() - last_report > 600:
            progress(list(running)); last_report = time.time()
        time.sleep(30)
    print("all done")

run_all()

seeds to run: [3, 4]
started seed 3 on GPU 0 at 16:13:11
started seed 4 on GPU 1 at 16:16:11
--- progress at 16:19:11 ---
seed 3: batch 552, loss 0.5558938384056091
seed 4: batch 552, loss 0.5722813606262207
--- progress at 16:49:11 ---
seed 3: batch 552, loss 0.5558938384056091
seed 4: batch 552, loss 0.5722813606262207
--- progress at 16:59:11 ---
seed 3: batch 552, loss 0.5558938384056091
seed 4: batch 552, loss 0.5722813606262207
--- progress at 17:09:11 ---
seed 3: batch 552, loss 0.5558938384056091
seed 4: batch 552, loss 0.5722813606262207
--- progress at 17:39:11 ---
seed 3: batch 552, loss 0.5933271646499634
seed 4: batch 552, loss 0.5314546823501587
--- progress at 17:49:11 ---
seed 3: batch 552, loss 0.5933271646499634
seed 4: batch 552, loss 0.5314546823501587
--- progress at 17:59:11 ---
seed 3: batch 552, loss 0.5933271646499634
seed 4: batch 552, loss 0.5314546823501587
--- progress at 18:09:11 ---
seed 3: batch 552, loss 0.5933271646499634
seed 4: batch 552, loss 0.5314

In [3]:
import re, csv, statistics as st
EPOCHS = 5
log_path = lambda s: f"{REPO_DIR}/logs/graphmixer_wiki_seed{s}.txt"

def last(pattern, text):
    m = re.findall(pattern, text); return float(m[-1]) if m else None

for s in range(1, 6):
    print(f"seed {s} log:", "found" if os.path.exists(log_path(s)) else "MISSING")
print("archive of old 1-epoch runs:", sorted(os.listdir(f"{REPO_DIR}/logs/archive_graphmixer_1epoch")) if os.path.isdir(f"{REPO_DIR}/logs/archive_graphmixer_1epoch") else "MISSING")

rows = []
for s in range(1, 6):
    p = log_path(s)
    if not os.path.exists(p): continue
    t = open(p, errors="ignore").read().replace("\r", "\n")
    val  = last(r"Validation: mrr:\s*([0-9.]+)", t)
    test = last(r"Test: mrr:\s*([0-9.]+)", t)
    auc  = last(r"train roc_auc,\s*([0-9.]+)", t)
    tv   = last(r"Total train & validation elapsed time \(s\):\s*([0-9.]+)", t)
    tt   = last(r"Test elapsed time \(s\):\s*([0-9.]+)", t)
    if val is None or test is None or auc is None:
        print(f"seed {s}: log incomplete, skipped"); continue
    rows.append((s, EPOCHS, auc, auc < 0.55, val, test, tv, tt))

with open(f"{REPO_DIR}/results/graphmixer_wiki.csv", "w") as f:
    f.write("seed,epochs_run,train_roc_auc,failed,val_mrr,test_mrr,train_val_time_s,test_time_s\n")
    for r in rows: f.write(",".join("" if x is None else str(x) for x in r) + "\n")
print(open(f"{REPO_DIR}/results/graphmixer_wiki.csv").read())

open(f"{REPO_DIR}/logs/graphmixer_source.txt", "w").write(
    "fpour/TGB_Baselines commit: d2022fae4a3311185f611bab733e267938251f42\n"
    f"epochs per seed: {EPOCHS}\n"
    "repo defaults: num_epochs=25, patience=10, test_interval_epochs=10\n"
    "seeds 1-2 ran on Kaggle account A, seeds 3-4 on account B (see logs/environment*.txt)\n"
    "published setup (TGB paper) does not state epochs\n")
save_and_push("GraphMixer tgbl-wiki summary CSV (seeds 1-4, 5 epochs)",
              ["results/graphmixer_wiki.csv", "logs/graphmixer_source.txt"])

seed 1 log: found
seed 2 log: found
seed 3 log: found
seed 4 log: found
seed 5 log: MISSING
archive of old 1-epoch runs: ['graphmixer_wiki_seed1.txt', 'graphmixer_wiki_seed2.txt', 'graphmixer_wiki_seed3.txt', 'graphmixer_wiki_seed4.txt', 'graphmixer_wiki_seed5.txt']
seed,epochs_run,train_roc_auc,failed,val_mrr,test_mrr,train_val_time_s,test_time_s
1,5,0.8559,False,0.1859,0.1892,19560.500511,3742.2356
2,5,0.5,True,0.002,0.002,19599.05181,3724.9255
3,5,0.8555,False,0.1644,0.1598,18766.514051,3775.1851
4,5,0.887,False,0.1402,0.1242,18814.577497,3756.5486


M  logs/graphmixer_source.txt
M  results/graphmixer_wiki.csv

[main 3192b23] GraphMixer tgbl-wiki summary CSV (seeds 1-4, 5 epochs)
 2 files changed, 3 insertions(+)

Current branch main is up to date.
From https://github.com/Rosierobber/Temporal-Graph-ML-Project
 * branch            main       -> FETCH_HEAD

To https://github.com/Rosierobber/Temporal-Graph-ML-Project.git
   0816347..3192b23  main -> main



In [4]:
TGB = f"{REPO_DIR}/third_party/TGB"
!date
!ls -la --time-style=full-iso {TGB}/examples/linkproppred/tgbl-review/saved_models/ 2>/dev/null
!ls -la {REPO_DIR}/results/perquery/ 2>/dev/null

Sun Nov 22 06:38:58 AM UTC 2026


In [5]:
import re, time, subprocess
from IPython.display import clear_output

def launch(script, seed, gpu, tag, cap=30):
    log = f"{REPO_DIR}/logs/{tag}_review_seed{seed}.txt"
    cmd = (f"cd {TGB} && PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES={gpu} PYTHONPATH=. "
           f"python -u examples/linkproppred/tgbl-review/{script} "
           f"--data tgbl-review --seed {seed} --num_epoch {cap} > {log} 2>&1")
    return subprocess.Popen(cmd, shell=True)

pB1 = launch("dyrep.py", 1, 0, "dyrep")
pB2 = launch("tgn.py",   2, 1, "tgn")
print("started", time.strftime("%H:%M:%S"))

started 06:39:18


In [7]:
# first confirm DyRep is alive and the dataset files exist
print("DyRep alive:", pB1.poll() is None)
!find {TGB}/tgb/datasets/tgbl_review -type f -exec ls -la {} \;

pB2 = launch("tgn.py", 2, 1, "tgn")
print("relaunched", time.strftime("%H:%M:%S"))

DyRep alive: False
find: ‘{TGB}/tgb/datasets/tgbl_review’: No such file or directory
relaunched 06:47:23


In [8]:
print("DyRep exit code:", pB1.poll())
for name in ["dyrep_review_seed1", "tgn_review_seed2"]:
    p = f"{REPO_DIR}/logs/{name}.txt"
    t = open(p, errors="ignore").read().replace("\r", "\n") if os.path.exists(p) else "(no log)"
    print(f"\n=== {name} (last 12 lines) ===")
    print("\n".join([l for l in t.splitlines() if l.strip()][-12:]))

print("\n=== dataset files ===")
root = f"{TGB}/tgb/datasets/tgbl_review"
for dp, _, fs in os.walk(root):
    for f in fs:
        fp = os.path.join(dp, f); print(f"{os.path.getsize(fp)/1e6:10.1f} MB  {os.path.relpath(fp, root)}")
print("TGN seed 2 alive:", pB2.poll() is None)

DyRep exit code: -2

=== dyrep_review_seed1 (last 12 lines) ===
    return forward_call(*args, **kwargs)
  File "/kaggle/working/Temporal-Graph-ML-Project/third_party/TGB/modules/emb_module.py", line 29, in forward
    return self.conv(x, edge_index, edge_attr)
           ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1779, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py", line 1790, in _call_impl
    return forward_call(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/torch_geometric/nn/conv/transformer_conv.py", line 196, in forward
    def forward(  # noqa: F811
KeyboardInterrupt

=== tgn_review_seed2 (last 12 lines) ===
INFO: Arguments: Namespace(data='tgbl-review', lr=0.0001, bs=200, k_value=10, num_epoch=30, seed=2, mem_dim=100, time_dim=100, emb_dim=100, tole

In [9]:
print("DyRep alive:", pB1.poll() is None, "| TGN alive:", pB2.poll() is None)

DyRep alive: False | TGN alive: True


In [13]:
!pkill -f "tgbl-review"
import shutil
shutil.rmtree(f"{TGB}/tgb/datasets/tgbl_review", ignore_errors=True)

def launch(script, seed, gpu, tag, cap=12):
    log = f"{REPO_DIR}/logs/{tag}_review_seed{seed}.txt"
    cmd = (f"cd {TGB} && PYTHONUNBUFFERED=1 CUDA_VISIBLE_DEVICES={gpu} PYTHONPATH=. "
           f"python -u examples/linkproppred/tgbl-review/{script} "
           f"--data tgbl-review --seed {seed} --num_epoch {cap} --num_run 1 > {log} 2>&1")
    return subprocess.Popen(cmd, shell=True, start_new_session=True)

pB2 = launch("tgn.py", 2, 1, "tgn")     # ONE job only; start the next after this one's first validation line appears